# Retail Promotions Analysis
**Codebasics Internship Project 1**

This notebook cleans the event data, joins the four source tables, and answers all ten client requests.

## Metric definitions

- **IR%** = `(revenue_after - revenue_before) / revenue_before * 100`
- **ISU%** = `(units_after - units_before) / units_before * 100`

Percentages are calculated from aggregated totals, not averaged from row-level percentages.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / 'datasets'
campaigns = pd.read_csv(DATA / 'dim_campaigns.csv')
products = pd.read_csv(DATA / 'dim_products.csv')
stores = pd.read_csv(DATA / 'dim_stores.csv')
events_raw = pd.read_csv(DATA / 'fact_events.csv')
{name: frame.shape for name, frame in {'campaigns': campaigns, 'products': products, 'stores': stores, 'events': events_raw}.items()}

{'campaigns': (2, 4),
 'products': (15, 3),
 'stores': (50, 2),
 'events': (1510, 9)}

## Data cleaning

In [2]:
business_key = ['store_id', 'campaign_id', 'product_code']
duplicates_removed = int(events_raw.duplicated(business_key).sum())
events = events_raw.drop_duplicates(business_key, keep='first').copy()
missing_filled = int(events['quantity_sold(before_promo)'].isna().sum())
median_used = float(events['quantity_sold(before_promo)'].median())
events['quantity_sold(before_promo)'] = events['quantity_sold(before_promo)'].fillna(median_used)
{'duplicates_removed': duplicates_removed, 'missing_filled': missing_filled, 'median_used': median_used, 'clean_rows': len(events)}

{'duplicates_removed': 10,
 'missing_filled': 20,
 'median_used': 78.0,
 'clean_rows': 1500}

In [3]:
data = (events.merge(campaigns, on='campaign_id', validate='many_to_one')
        .merge(products, on='product_code', validate='many_to_one')
        .merge(stores, on='store_id', validate='many_to_one'))
data['revenue_before'] = data['base_price(before_promo)'] * data['quantity_sold(before_promo)']
data['revenue_after'] = data['base_price(after_promo)'] * data['quantity_sold(after_promo)']
data.shape

(1500, 17)

## Client requests 1-4: quality, store coverage, and pricing

In [4]:
city_counts = stores.groupby('city')['store_id'].nunique().sort_values(ascending=False)
category_prices = data.groupby('category')['base_price(before_promo)'].agg(['min', 'mean']).sort_values('mean')
print('Q1 duplicates removed:', duplicates_removed)
print('Q2 cities with >5 stores:', int((city_counts > 5).sum()))
print('Q3 missing filled:', missing_filled, '| median:', median_used)
print('Q4 lowest-price category:', category_prices.index[0])
display(city_counts.to_frame('store_count'), category_prices)

Q1 duplicates removed: 10
Q2 cities with >5 stores: 3
Q3 missing filled: 20 | median: 78.0
Q4 lowest-price category: Personal Care


,store_count
city,
Bengaluru,10
Chennai,8
Hyderabad,7
Coimbatore,5
Visakhapatnam,5
Madurai,4
Mysuru,4
Mangalore,3
Trivandrum,2


,min,mean
category,,
Personal Care,50,102.375
Grocery & Staples,156,385.000
Home Care,55,490.000
Home Appliances,350,685.000
Combo1,3000,3000.000


## Client requests 5-7: Diwali stores and campaign comparison

In [5]:
diwali = data.query("campaign_name == 'Diwali'")
q5 = diwali.query("promo_type == 'BOGOF'")['quantity_sold(after_promo)'].sum()
diwali_stores = diwali.groupby(['store_id', 'city'])['quantity_sold(after_promo)'].sum().sort_values(ascending=False)
campaign_perf = data.groupby('campaign_name').agg(units_before=('quantity_sold(before_promo)', 'sum'), units_after=('quantity_sold(after_promo)', 'sum'))
campaign_perf['unit_increase'] = campaign_perf['units_after'] - campaign_perf['units_before']
campaign_perf['isu_pct'] = campaign_perf['unit_increase'] / campaign_perf['units_before'] * 100
print('Q5 Diwali BOGOF units:', int(q5))
print('Q6 top Diwali store:', diwali_stores.index[0], '| units:', int(diwali_stores.iloc[0]))
print('Q7 greater increase:', campaign_perf['unit_increase'].idxmax())
display(campaign_perf)

Q5 Diwali BOGOF units: 34461
Q6 top Diwali store: ('STCHE-4', 'Chennai') | units: 5013
Q7 greater increase: Sankranti


,units_before,units_after,unit_increase,isu_pct
campaign_name,,,,
Diwali,109756.0,183404,73648.0,67.101571
Sankranti,97894.0,252069,154175.0,157.491777


## Client request 8: highest Sankranti product IR%

In [6]:
sankranti = data.query("campaign_name == 'Sankranti'")
product_ir = sankranti.groupby(['product_code', 'product_name']).agg(revenue_before=('revenue_before', 'sum'), revenue_after=('revenue_after', 'sum'))
product_ir['ir_pct'] = (product_ir['revenue_after'] - product_ir['revenue_before']) / product_ir['revenue_before'] * 100
product_ir.sort_values('ir_pct', ascending=False).head(10)

,,revenue_before,revenue_after,ir_pct
product_code,product_name,,,
P03,Atliq_Suflower_Oil (1L),3189600.0,6118500,91.826561
P15,Atliq_Home_Essential_8_Product_Combo,16185000.0,31027500,91.705283
P13,Atliq_High_Glo_15W_LED_Bulb,1740550.0,3303125,89.774784
P14,Atliq_waterproof_Immersion_Rod,4542060.0,8534850,87.907029
P04,Atliq_Farm_Chakki_Atta (1KG),6813550.0,12779800,87.564485
P08,Atliq_Double_Bedsheet_set,2485910.0,4649330,87.027286
P07,Atliq_Curtains,583500.0,1065450,82.596401
P01,Atliq_Masoor_Dal (1KG),2108376.0,2011235,-4.607385
P02,Atliq_Sonamasuri_Rice (10KG),18374760.0,17262720,-6.051997


## Client request 9: lowest Diwali ISU% in Visakhapatnam

In [7]:
vizag_isu = diwali.query("city == 'Visakhapatnam'").groupby('store_id').agg(units_before=('quantity_sold(before_promo)', 'sum'), units_after=('quantity_sold(after_promo)', 'sum'))
vizag_isu['isu_pct'] = (vizag_isu['units_after'] - vizag_isu['units_before']) / vizag_isu['units_before'] * 100
vizag_isu.sort_values('isu_pct')

,units_before,units_after,isu_pct
store_id,,,
STVSK-3,1780.0,2656,49.213483
STVSK-4,1926.0,2908,50.986501
STVSK-1,1903.0,3078,61.744614
STVSK-2,1701.0,2860,68.136390
STVSK-0,1768.0,3005,69.966063


## Client request 10: negative Sankranti promotion types

In [8]:
promo_perf = sankranti.groupby('promo_type').agg(revenue_before=('revenue_before', 'sum'), revenue_after=('revenue_after', 'sum'), units_before=('quantity_sold(before_promo)', 'sum'), units_after=('quantity_sold(after_promo)', 'sum'))
promo_perf['ir_pct'] = (promo_perf['revenue_after'] - promo_perf['revenue_before']) / promo_perf['revenue_before'] * 100
promo_perf['isu_pct'] = (promo_perf['units_after'] - promo_perf['units_before']) / promo_perf['units_before'] * 100
display(promo_perf.sort_values('ir_pct'))
print('Q10:', promo_perf.query('ir_pct < 0 and isu_pct < 0').index.tolist())

,revenue_before,revenue_after,units_before,units_after,ir_pct,isu_pct
promo_type,,,,,,
25% OFF,935195.0,567387,6601.0,5307,-39.329552,-19.603090
50% OFF,527482.0,363564,4451.0,6100,-31.075563,37.047854
33% OFF,20483136.0,19273955,33624.0,47459,-5.903300,41.146205
BOGOF,19355170.0,36451055,47823.0,180792,88.327227,278.044037
500 Cashback,16185000.0,31027500,5395.0,12411,91.705283,130.046339


Q10: ['25% OFF']


## Conclusions

- Sankranti generated the larger incremental unit volume.
- BOGOF materially increased unit movement during Diwali.
- 25% OFF under Sankranti reduced both revenue and units, making it the clearest candidate for review.
- The workbook in `outputs/retail_promotions_analysis.xlsx` provides the presentation-ready dashboard and supporting tables.